In [1]:
import ollama

In [2]:
#dummy database
inventory_db={
    "laptop":{"stock":5,"base_price":1200},
    "monitor":{"stock":0,"base_price":300},
    "keyboard":{"stock":25,"base_price":80}}


In [3]:
#defining tool
#tool1 :checking the tool in the databse 
def check_inventory(product_name):
    product_name=product_name.lower()
    if product_name in inventory_db:
        return inventory_db[product_name]
    return {"stock":0,"base_price":None}

#tool2:business logic for discounts
def calculate_loyalty_discount(base_price,years_as_customer):
    discount=min(years_as_customer*0.05,0.30)
    final_price=base_price*(1-discount)
    return round(final_price,2)
    

In [11]:
#mapping the function so the script can call by name 
available_functions={
    'check_inventory':check_inventory,
    'calculate_loyalty_discount':calculate_loyalty_discount}

In [5]:
#defining the schema
#DEFINING SCHEMA

tools = [
    {
        "type": "function",
        "function": {
            "name": "check_inventory",
            "description": "Get stock and price for a product",
            "parameters": {
                "type": "object",
                "properties": {
                    "product_name": {"type": "string"}
                },
                "required": ["product_name"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_loyalty_discount",
            "description": "Calculate final price based on loyalty years",
            "parameters": {
                "type": "object",
                "properties": {
                    "base_price": {"type": "number"},
                    "years_as_customer": {"type": "integer"}
                },
                "required": ["base_price", "years_as_customer"]
            }
        }
    }
]

In [6]:
tools

[{'type': 'function',
  'function': {'name': 'check_inventory',
   'description': 'Get stock and price for a product',
   'parameters': {'type': 'object',
    'properties': {'product_name': {'type': 'string'}},
    'required': ['product_name']}}},
 {'type': 'function',
  'function': {'name': 'calculate_loyalty_discount',
   'description': 'Calculate final price based on loyalty years',
   'parameters': {'type': 'object',
    'properties': {'base_price': {'type': 'number'},
     'years_as_customer': {'type': 'integer'}},
    'required': ['base_price', 'years_as_customer']}}}]

In [17]:
message = [
    {
        "role": "user",
        "content": "I am the customer from 5 years. What will be the final price for laptop for me?"
    }
]

while True:

    # Call the LLM
    response = ollama.chat(
        model="qwen3:4b",
        tools=tools,
        messages=message
    )

    # Add LLM response to conversation
    message.append(response["message"])

    # Check whether LLM wants to call a tool
    tool_calls = response["message"].get("tool_calls")

    # No tool call = final answer
    if not tool_calls:
        print(response["message"]["content"])
        break

    # Execute the requested tools
    for tool_call in tool_calls:

        tool_name = tool_call["function"]["name"]
        tool_args = tool_call["function"]["arguments"]

        # Find the actual Python function
        function_call = available_functions[tool_name]

        # Execute the function
        result = function_call(**tool_args)

        print("Tool:", tool_name)
        print("Result:", result)

        # Send tool result back to the LLM
        message.append({
            "role": "tool",
            "content": str(result)
        })

Tool: check_inventory
Result: {'stock': 5, 'base_price': 1200}
Tool: calculate_loyalty_discount
Result: 900.0
The final price for the laptop after applying your 5 years of loyalty is **$900.00**.
